In [1]:
# === Stage2FE : CELL 1 (REPLACEMENT) ===
# Purpose:
#  - Load Stage-2 master (v3 preferred), verify years and ordering.
#  - Save df_stage2_sorted.csv (chronological master) and temporal splits.
#  - Create Tier folders and write master + train/val/test CSVs for each Tier.
#  - Print diagnostics (counts, unique years, round distribution, sample rows).
#
# Run this from your project root (BASE set below). It is defensive: it will
# pick v3 if available, otherwise fallback to existing master file names.

import os
from pathlib import Path
import json
import pandas as pd
import numpy as np

# --------- Paths (adjust if your layout differs) ----------
BASE = Path(r"D:\Courses\Global Academy of Technology\kcet-college-pred\Version2")
STAGE2_OUT = BASE / "data" / "stage2_outputs"
# prefer the v3 corrected master if present
POSSIBLE_MASTERS = [
    STAGE2_OUT / "KCET_stage2_final_with_encoders_v3.csv",
    STAGE2_OUT / "KCET_stage2_final_with_encoders_v2.csv",
    STAGE2_OUT / "KCET_stage2_final_with_encoders.csv",
    STAGE2_OUT / "KCET_stage2_final_with_encoders_v3.xlsx"
]
MASTER = None
for p in POSSIBLE_MASTERS:
    if p.exists():
        MASTER = p
        break
if MASTER is None:
    raise FileNotFoundError("No Stage-2 master file found in stage2_outputs. Checked: " + ", ".join(str(x.name) for x in POSSIBLE_MASTERS))

print("Using Stage-2 input file:", MASTER)
df = pd.read_csv(MASTER)

print("Loaded dataframe shape:", df.shape)
# show basic column list (truncated)
print("Columns (preview):", df.columns.tolist()[:40])

# Ensure Year column exists and is int
if "Year" not in df.columns:
    raise RuntimeError("Stage-2 master missing required column 'Year'")

df['Year'] = df['Year'].astype(int)

# Sort master chronologically by Year then Round (keep stable order)
if "Round" in df.columns:
    df = df.sort_values(["Year","Round"], ascending=[True, True]).reset_index(drop=True)
else:
    df = df.sort_values(["Year"], ascending=True).reset_index(drop=True)

# Save a sorted master snapshot for downstream cells
OUT_MASTER_SORTED = STAGE2_OUT / "df_stage2_sorted.csv"
df.to_csv(OUT_MASTER_SORTED, index=False)
print("Saved sorted master ->", OUT_MASTER_SORTED)

# Basic year checks and split years (default temporal split used in the project)
train_years = [2020, 2021, 2022]
val_year = 2023
test_year = 2024

unique_years = sorted(df['Year'].unique().tolist())
print("Years in data:", unique_years)

# sanity: ensure years for split exist
for y in train_years + [val_year, test_year]:
    if y not in unique_years:
        print(f"WARNING: expected year {y} not present in data (present: {unique_years})")

# create split masks (strict year-based)
train_df = df[df['Year'].isin(train_years)].copy().reset_index(drop=True)
val_df   = df[df['Year'] == val_year].copy().reset_index(drop=True)
test_df  = df[df['Year'] == test_year].copy().reset_index(drop=True)

print("Split sizes — train:", len(train_df), "val:", len(val_df), "test:", len(test_df))

# quick distribution diagnostics
def print_year_round_stats(dset, name):
    yrs = sorted(dset['Year'].unique().tolist())
    rounds = sorted(dset['Round'].unique().tolist()) if 'Round' in dset.columns else None
    print(f"  {name}: rows={len(dset)}, years={yrs}, rounds_sample={rounds}")
    # sample target stats if present
    if 'Cutoff_Rank' in dset.columns:
        t = dset['Cutoff_Rank'].dropna()
        print(f"    Cutoff_Rank: mean={t.mean():.1f}, median={t.median():.1f}, min={t.min():.1f}, max={t.max():.1f}")

print_year_round_stats(train_df, "TRAIN")
print_year_round_stats(val_df, "VAL")
print_year_round_stats(test_df, "TEST")

# Tiered CSV outputs: ensure Tier column exists
if "College_Tier" not in df.columns:
    print("WARNING: 'College_Tier' not found in master — tier CSVs will not be created.")
else:
    tiers = sorted(df['College_Tier'].dropna().unique().tolist())
    print("Tiers found:", tiers)
    for t in tiers:
        tier_folder = STAGE2_OUT / f"Tier{t}"
        tier_folder.mkdir(parents=True, exist_ok=True)
        # master for this tier
        tier_master = df[df['College_Tier']==t].copy().reset_index(drop=True)
        tier_master_file = tier_folder / f"KCET_stage2_Tier{t}.csv"
        tier_master.to_csv(tier_master_file, index=False)
        # temporal splits for this tier
        tier_train = tier_master[tier_master['Year'].isin(train_years)].reset_index(drop=True)
        tier_val   = tier_master[tier_master['Year']==val_year].reset_index(drop=True)
        tier_test  = tier_master[tier_master['Year']==test_year].reset_index(drop=True)
        tier_train.to_csv(tier_folder / "tier1_train.csv" if t==1 else tier_folder / f"tier{t}_train.csv", index=False)
        tier_val.to_csv(tier_folder / "tier1_val.csv" if t==1 else tier_folder / f"tier{t}_val.csv", index=False)
        tier_test.to_csv(tier_folder / "tier1_test.csv" if t==1 else tier_folder / f"tier{t}_test.csv", index=False)
        print(f"Saved Tier{t} CSVs: master={tier_master_file} train={len(tier_train)} val={len(tier_val)} test={len(tier_test)}")

# also save global train/val/test in stage2_outputs (non-tiered)
(train_df).to_csv(STAGE2_OUT / "split_train_raw.csv", index=False)
(val_df).to_csv(STAGE2_OUT / "split_val_raw.csv", index=False)
(test_df).to_csv(STAGE2_OUT / "split_test_raw.csv", index=False)
print("Saved global temporal splits: split_train_raw.csv, split_val_raw.csv, split_test_raw.csv")

# Save a small JSON manifest for downstream cells / model training
manifest = {
    "master_path": str(MASTER),
    "sorted_master": str(OUT_MASTER_SORTED),
    "split_train": str(STAGE2_OUT / "split_train_raw.csv"),
    "split_val":   str(STAGE2_OUT / "split_val_raw.csv"),
    "split_test":  str(STAGE2_OUT / "split_test_raw.csv"),
    "train_years": train_years,
    "val_year": val_year,
    "test_year": test_year,
    "rows_master": int(len(df)),
    "rows_train": int(len(train_df)),
    "rows_val": int(len(val_df)),
    "rows_test": int(len(test_df))
}
with open(STAGE2_OUT / "stage2_final_manifest_checked.json", "w", encoding="utf8") as f:
    json.dump(manifest, f, indent=2)
print("Saved manifest ->", STAGE2_OUT / "stage2_final_manifest_checked.json")

# Print a small sample to inspect
print("\n--- SAMPLE ROWS (master head) ---")
display(df.head(6))

print("\nCELL 1 complete — temporal splits created and saved. Paste this cell output back here and I'll provide CELL 2 (feature engineering) that computes causal/historical features without leakage.")


Using Stage-2 input file: D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\stage2_outputs\KCET_stage2_final_with_encoders_v3.csv
Loaded dataframe shape: (212487, 40)
Columns (preview): ['College_Code', 'College_Name', 'Category', 'Branch', 'Cutoff_Rank', 'Cutoff_log1p', 'Year', 'Round', 'Exam_Type', 'College_Tier', 'College_Code_prev1_median', 'College_Code_prev3_median', 'College_Code_count_prev', 'College_Code_trend_prev3_slope', 'Branch_prev1_median', 'Branch_prev3_median', 'Branch_count_prev', 'Branch_trend_prev3_slope', 'Category_prev1_median', 'Category_prev3_median', 'Category_count_prev', 'Category_trend_prev3_slope', 'College_Code_prev1_median_ranknorm', 'College_Code_prev3_median_ranknorm', 'Branch_prev1_median_ranknorm', 'Branch_prev3_median_ranknorm', 'Category_prev1_median_ranknorm', 'Category_prev3_median_ranknorm', 'College_Code_enc_prev_filled_v3', 'Branch_enc_prev_filled_v3', 'Category_enc_prev_filled_v3', 'college_pop_prev_filled', 'branch_pop_p

,College_Code,College_Name,Category,Branch,Cutoff_Rank,Cutoff_log1p,Year,Round,Exam_Type,College_Tier,...,Category_enc_prev_filled_v3,college_pop_prev_filled,branch_pop_prev_filled,college_pop_prev_norm,branch_pop_prev_norm,branch_unstable_flag_v3,tier_prev1_median,tier_prev1_median_ranknorm,Rank_norm,Rank_norm_log1p
0,E001,University Visveswariah College of Engineering...,1G,CE Civil,29057.0,10.277049,2020,1,KCET,1,...,0.274636,0,0,0.0,0.0,0,10.449381,0.224952,0.189341,0.173399
1,E001,University Visveswariah College of Engineering...,1K,CE Civil,30427.0,10.323119,2020,1,KCET,1,...,0.342367,0,0,0.0,0.0,0,10.449381,0.224952,0.198268,0.180877
2,E001,University Visveswariah College of Engineering...,2AG,CE Civil,27430.0,10.219429,2020,1,KCET,1,...,0.272987,0,0,0.0,0.0,0,10.449381,0.224952,0.178739,0.164445
3,E001,University Visveswariah College of Engineering...,2AR,CE Civil,28173.0,10.246155,2020,1,KCET,1,...,0.315315,0,0,0.0,0.0,0,10.449381,0.224952,0.183581,0.168544
4,E001,University Visveswariah College of Engineering...,2BG,CE Civil,26255.0,10.175650,2020,1,KCET,1,...,0.286499,0,0,0.0,0.0,0,10.449381,0.224952,0.171082,0.157929
5,E001,University Visveswariah College of Engineering...,2BR,CE Civil,50152.0,10.822834,2020,1,KCET,1,...,0.343068,0,0,0.0,0.0,0,10.449381,0.224952,0.326800,0.282770



CELL 1 complete — temporal splits created and saved. Paste this cell output back here and I'll provide CELL 2 (feature engineering) that computes causal/historical features without leakage.


In [2]:
# === Stage-2 : CELL 2 (FEATURE ENGINEERING — causal / leak-free) ===
# Run after CELL 1 (df_stage2_sorted.csv produced).
import math
import json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from collections import defaultdict

BASE = Path(r"D:\Courses\Global Academy of Technology\kcet-college-pred\Version2")
STAGE2_OUT = BASE / "data" / "stage2_outputs"

# Files
SORTED_MASTER = STAGE2_OUT / "df_stage2_sorted.csv"
if not SORTED_MASTER.exists():
    raise FileNotFoundError("Sorted master not found: " + str(SORTED_MASTER))

df = pd.read_csv(SORTED_MASTER)
print("Loaded sorted master:", SORTED_MASTER, "shape:", df.shape)

# load year_max_ranks if present to compute rank-norm
yr_max_file = STAGE2_OUT / "year_max_ranks.json"
if yr_max_file.exists():
    year_max = json.load(open(yr_max_file, "r"))
    # convert keys to int if strings
    year_max = {int(k): float(v) for k, v in year_max.items()}
    print("Loaded year_max_ranks for years:", sorted(year_max.keys()))
else:
    # fallback: use empirical max per year from df
    year_max = dict(df.groupby("Year")["Cutoff_Rank"].max().astype(float))
    print("year_max_ranks not found — computed from data:", {k:int(v) for k,v in year_max.items()})

# ensure target present
if "Cutoff_log1p" not in df.columns:
    if "Cutoff_Rank" not in df.columns:
        raise RuntimeError("Need Cutoff_Rank to compute Cutoff_log1p")
    df["Cutoff_log1p"] = np.log1p(df["Cutoff_Rank"].astype(float))

# helper: global fallback median
global_median = float(df.loc[df['Year'] < df['Year'].max(), 'Cutoff_log1p'].median()) if 'Year' in df.columns else float(df['Cutoff_log1p'].median())
print("Global target median (for fallback):", global_median)

entities = ["College_Code", "Branch", "Category"]
target_col = "Cutoff_log1p"
rankcol = "Cutoff_Rank"

# Precompute per-entity/year medians to make lookups efficient
# Structure: med_map[entity][entity_value][year] = median
med_map = {e: defaultdict(dict) for e in entities}
rank_med_map = {e: defaultdict(dict) for e in entities}
count_map = {e: defaultdict(int) for e in entities}

for e in entities:
    g = df.groupby([e, "Year"])[target_col].median().reset_index().rename(columns={target_col:"yr_median"})
    for _, row in g.iterrows():
        med_map[e][row[e]][int(row["Year"])] = float(row["yr_median"])
    # rank-norm medians per year as well
    if rankcol in df.columns:
        g2 = df.groupby([e, "Year"])[rankcol].median().reset_index().rename(columns={rankcol:"yr_median_rank"})
        for _, row in g2.iterrows():
            y = int(row["Year"])
            maxr = year_max.get(y, np.nan)
            if np.isfinite(maxr) and maxr>0:
                rank_med_map[e][row[e]][y] = float(row["yr_median_rank"]) / float(maxr)
            else:
                rank_med_map[e][row[e]][y] = float(row["yr_median_rank"])
    # counts prior per entity
    count_map[e] = df.groupby(e).size().to_dict()

# function to compute prior aggregates for a single entity value and current year
def compute_prior_stats(entity, value, year):
    """
    returns dict with:
      prev1_median, prev3_median, count_prev, trend_prev3_slope
      prev1_median_ranknorm, prev3_median_ranknorm
    All computed using only years < current year.
    """
    years = sorted([y for y in med_map[entity][value].keys() if y < year])
    out = {}
    if len(years) == 0:
        out['prev1_median'] = np.nan
        out['prev3_median'] = np.nan
        out['count_prev'] = 0
        out['trend_prev3_slope'] = np.nan
        out['prev1_median_ranknorm'] = np.nan
        out['prev3_median_ranknorm'] = np.nan
        return out

    # prev1 = median of the latest prior year
    last_year = years[-1]
    out['prev1_median'] = med_map[entity][value][last_year]
    # prev3 = median of last up to 3 prior years (year-wise medians averaged)
    last_k = years[-3:]  # last up to 3 prior years
    prev3_vals = [med_map[entity][value][y] for y in last_k]
    out['prev3_median'] = float(np.median(prev3_vals)) if len(prev3_vals)>0 else np.nan

    # count_prev = number of rows for this entity with year < current year
    # (we approximate using groupby counts by year from df)
    # compute by summing counts for those years
    cnt = 0
    sub = df[(df[entity]==value) & (df['Year'] < year)]
    cnt = len(sub)
    out['count_prev'] = int(cnt)

    # trend: linear slope fit on (years, per-year median) for those last up to 3 years
    if len(last_k) >= 2:
        xs = np.array(last_k, dtype=float)
        ys = np.array([med_map[entity][value][y] for y in last_k], dtype=float)
        # normalize xs to small range to keep slope scale comparable
        xs_norm = xs - xs.mean()
        try:
            slope = np.polyfit(xs_norm, ys, 1)[0]
        except Exception:
            slope = 0.0
        out['trend_prev3_slope'] = float(slope)
    else:
        out['trend_prev3_slope'] = 0.0

    # ranknorm versions
    rank_years = sorted([y for y in rank_med_map[entity][value].keys() if y < year])
    if len(rank_years)==0:
        out['prev1_median_ranknorm'] = np.nan
        out['prev3_median_ranknorm'] = np.nan
    else:
        last_ry = rank_years[-1]
        out['prev1_median_ranknorm'] = rank_med_map[entity][value].get(last_ry, np.nan)
        last_k_ry = rank_years[-3:]
        vals_ry = [rank_med_map[entity][value].get(y, np.nan) for y in last_k_ry]
        vals_ry = [v for v in vals_ry if not np.isnan(v)]
        out['prev3_median_ranknorm'] = float(np.median(vals_ry)) if len(vals_ry)>0 else np.nan

    return out

# We'll iterate rows and compute features — vectorize by caching entity-year results to avoid repeated compute
cache_prior = {}

# columns to add
new_cols = []
for e in entities:
    new_cols += [
        f"{e}_prev1_median",
        f"{e}_prev3_median",
        f"{e}_count_prev",
        f"{e}_trend_prev3_slope",
        f"{e}_prev1_median_ranknorm",
        f"{e}_prev3_median_ranknorm",
    ]

# Add safe existing enc columns if present and plan to create v4 encoders
enc_out = {e: {} for e in entities}  # entity -> dict(entity_value -> encoder_value)
unstable_candidates = {}

# Prepare output columns and fill with NaNs / defaults
for c in new_cols:
    df[c] = np.nan

# iterate rows (vectorized-ish by using med_map) — this is safe (uses only prior-year medians)
years_sorted = sorted(df['Year'].unique())
for idx, row in df.iterrows():
    year = int(row['Year'])
    for e in entities:
        val = row[e]
        key = (e, val, year)
        if key in cache_prior:
            stats = cache_prior[key]
        else:
            stats = compute_prior_stats(e, val, year)
            cache_prior[key] = stats
        # fill df
        df.at[idx, f"{e}_prev1_median"] = stats.get('prev1_median', np.nan)
        df.at[idx, f"{e}_prev3_median"] = stats.get('prev3_median', np.nan)
        df.at[idx, f"{e}_count_prev"] = stats.get('count_prev', 0)
        df.at[idx, f"{e}_trend_prev3_slope"] = stats.get('trend_prev3_slope', 0.0)
        df.at[idx, f"{e}_prev1_median_ranknorm"] = stats.get('prev1_median_ranknorm', np.nan)
        df.at[idx, f"{e}_prev3_median_ranknorm"] = stats.get('prev3_median_ranknorm', np.nan)

# Now compute temporal encoders per entity: encoder = prior-median of target (all prior years combined)
for e in entities:
    enc_map = {}
    for val, years_dict in med_map[e].items():
        # collect all medians for years < max_year (we'll later use per-row year to pick only prior years)
        # But for encoder we compute per-year encoder later; here we'll compute an overall prior-median function per (val, year)
        pass

# Build per-entity per-year encoder: for each distinct (entity,value,year) we compute encoder = median of target for that entity across years < year
temporal_encoders = {e:{} for e in entities}
for e in entities:
    vals = df[e].unique()
    for val in vals:
        temporal_encoders[e].setdefault(val, {})
        years_present = sorted(df.loc[df[e]==val, 'Year'].unique().tolist())
        for y in years_present:
            # compute prior values for this entity where Year < y
            prior_vals = df[(df[e]==val) & (df['Year'] < y)][target_col].values
            if len(prior_vals) == 0:
                encoder_val = np.nan
            else:
                encoder_val = float(np.median(prior_vals))
            temporal_encoders[e][val][int(y)] = encoder_val

# Now create per-row filled encoder column using temporal_encoders: encoder is median of prior years (Year < row_year)
for idx, row in df.iterrows():
    y = int(row['Year'])
    for e in entities:
        val = row[e]
        enc_dict = temporal_encoders[e].get(val, {})
        # find last encoder available for years < y
        prior_years = sorted([yy for yy in enc_dict.keys() if int(yy) < y])
        if prior_years:
            last_y = prior_years[-1]
            enc_val = enc_dict[last_y]
        else:
            enc_val = np.nan
        df.at[idx, f"{e}_enc_prev_filled_v4"] = enc_val

# fill encoders with global median fallback (train-only global median)
train_years = [2020,2021,2022]
train_mask = df['Year'].isin(train_years)
global_median_train = float(df.loc[train_mask, target_col].median()) if train_mask.sum()>0 else global_median

for e in entities:
    col = f"{e}_enc_prev_filled_v4"
    df[col] = df[col].fillna(global_median_train)
    # build encoder dict mapping entity->overall prior-median (for deploy): use median of all prior values across train years
    enc_map = {}
    for val in df[e].unique():
        # encoder for deployment: median of target for val across all train years only (strict)
        prior_vals = df[(df[e]==val) & (df['Year'].isin(train_years))][target_col].values
        if len(prior_vals)==0:
            enc_map[val] = global_median_train
        else:
            enc_map[val] = float(np.median(prior_vals))
    temporal_encoders[e + "_final"] = enc_map  # note key suffix so we keep per-year encoders and final fallback enc map

# branch_unstable_flag_v4: mark branch as unstable if prior stddev of prev1/prev3 medians > threshold
# We'll compute for each Branch over train years standard deviation of per-year medians; if stddev_ranknorm > threshold mark unstable
unstable_flags = {}
threshold_std = 0.7  # you can adjust; conservative threshold to flag volatile branches
for val in df['Branch'].unique():
    yrs = sorted([y for y in med_map['Branch'][val].keys() if y in train_years])
    vals_med = [med_map['Branch'][val][y] for y in yrs]
    if len(vals_med) <= 1:
        unstable_flags[val] = 0
    else:
        stdv = float(np.std(vals_med))
        unstable_flags[val] = 1 if stdv >= threshold_std else 0

df['branch_unstable_flag_v4'] = df['Branch'].map(unstable_flags).fillna(0).astype(int)

# Rank normalisation: compute Rank_norm if not present
if 'Rank_norm' not in df.columns or df['Rank_norm'].isnull().all():
    df['Rank_norm'] = df.apply(lambda r: (r[rankcol]/year_max.get(int(r['Year']), r[rankcol])) if (pd.notnull(r[rankcol]) and int(r['Year']) in year_max) else np.nan, axis=1)
# also Rank_norm_log1p
df['Rank_norm_log1p'] = np.log1p(df['Rank_norm'].fillna(0.0))

# final set of candidate features (safe)
candidate_features = []
# per-entity features
for e in entities:
    candidate_features += [
        f"{e}_prev1_median",
        f"{e}_prev3_median",
        f"{e}_count_prev",
        f"{e}_trend_prev3_slope",
        f"{e}_prev1_median_ranknorm",
        f"{e}_prev3_median_ranknorm",
        f"{e}_enc_prev_filled_v4"
    ]
# other safe features
candidate_features += [
    "college_pop_prev_filled",
    "branch_pop_prev_filled",
    "college_pop_prev_norm" if 'college_pop_prev_norm' in df.columns else 'college_pop_prev_filled',
    "branch_pop_prev_norm" if 'branch_pop_prev_norm' in df.columns else 'branch_pop_prev_filled',
    "branch_unstable_flag_v4",
    "College_Tier",
    "Round",
    "Rank_norm",
    "Rank_norm_log1p"
]

# Ensure all candidate columns exist (create if missing default)
for c in candidate_features:
    if c not in df.columns:
        df[c] = np.nan

# Scaling: fit scaler on train set only for numeric scaler_cols
scaler_cols = [c for c in candidate_features if c not in ("College_Tier","Round","branch_unstable_flag_v4")]
# keep only numeric
scaler_cols = [c for c in scaler_cols if c in df.columns]
# fillna for scaler fitting on train
train_mask = df['Year'].isin(train_years)
scaler = StandardScaler()
# replace inf/nan with median (train median) to be safe
train_for_scaler = df.loc[train_mask, scaler_cols].copy()
for col in scaler_cols:
    col_med = train_for_scaler[col].median(skipna=True)
    if np.isnan(col_med):
        col_med = 0.0
    train_for_scaler[col] = train_for_scaler[col].fillna(col_med).astype(float)
scaler.fit(train_for_scaler.values)

# attach scaled columns for all rows
for col in scaler_cols:
    filled = df[col].fillna(df.loc[train_mask, col].median() if train_mask.sum()>0 else 0.0).astype(float)
    scaled_vals = scaler.transform(filled.values.reshape(-1,1)) if len(scaler_cols)==1 else None
# Because sklearn scaler was fit on multiple columns together, transform using full matrix
full_vals = df[scaler_cols].fillna(0.0).astype(float).values
scaled_all = scaler.transform(full_vals)
for i, col in enumerate(scaler_cols):
    df[col + "_scaled"] = scaled_all[:, i]

# Build final_features list (scaled numeric cols + discrete flags)
final_features = [c + "_scaled" for c in scaler_cols] + ["branch_unstable_flag_v4", "College_Tier", "Round"]
# dedupe
final_features = list(dict.fromkeys(final_features))

print("Candidate features prepared (count):", len(candidate_features))
print("Final model-ready features (count):", len(final_features))
print("Sample final features:", final_features[:12])

# Save outputs
OUT_MASTER_V4 = STAGE2_OUT / "KCET_stage2_final_with_encoders_v4.csv"
df.to_csv(OUT_MASTER_V4, index=False)
print("Saved Stage-2 master with v4 encoders ->", OUT_MASTER_V4)

# Save per-tier CSVs (use the same tiers from CELL1)
if "College_Tier" in df.columns:
    tiers = sorted(df['College_Tier'].dropna().unique().tolist())
    for t in tiers:
        tfolder = STAGE2_OUT / f"Tier{int(t)}"
        tfolder.mkdir(parents=True, exist_ok=True)
        tmaster = df[df['College_Tier']==t].copy().reset_index(drop=True)
        tmaster.to_csv(tfolder / f"KCET_stage2_Tier{int(t)}_v4.csv", index=False)
        train_t = tmaster[tmaster['Year'].isin(train_years)]
        val_t = tmaster[tmaster['Year']==2023]
        test_t = tmaster[tmaster['Year']==2024]
        train_t.to_csv(tfolder / f"tier{int(t)}_train_v4.csv", index=False)
        val_t.to_csv(tfolder / f"tier{int(t)}_val_v4.csv", index=False)
        test_t.to_csv(tfolder / f"tier{int(t)}_test_v4.csv", index=False)
        print(f"Saved Tier{int(t)} v4 CSVs: train={len(train_t)} val={len(val_t)} test={len(test_t)}")

# Save temporal encoders (we'll save per-entity per-year map + final train-only encoder map)
temporal_out_file = STAGE2_OUT / "temporal_encoders_v4.pkl"
joblib.dump(temporal_encoders, temporal_out_file)
print("Saved temporal encoders (per-entity per-year) ->", temporal_out_file)

# Save final per-entity final encoder (deployment fallback) as separate file
# temporal_encoders already contains keys like 'College_Code_final' if earlier created; we'll create clean final map now
final_encoder_map = {}
for e in entities:
    final_encoder_map[e] = {}
    for val in df[e].unique():
        prior_vals_train = df[(df[e]==val) & (df['Year'].isin(train_years))][target_col].values
        if len(prior_vals_train) == 0:
            final_encoder_map[e][val] = global_median_train
        else:
            final_encoder_map[e][val] = float(np.median(prior_vals_train))
joblib.dump(final_encoder_map, STAGE2_OUT / "temporal_encoders_final_v4.pkl")
print("Saved final encoder map ->", STAGE2_OUT / "temporal_encoders_final_v4.pkl")

# Save unstable branch list
unstable_list_v4 = [b for b,f in unstable_flags.items() if f==1]
with open(STAGE2_OUT / "unstable_branches_list_v4.json","w",encoding="utf8") as fo:
    json.dump(unstable_list_v4, fo, indent=2)
print("Saved unstable_branches_list_v4.json (count):", len(unstable_list_v4))

# Save scaler & final feature list & manifest
joblib.dump(scaler, STAGE2_OUT / "stage2_scaler_v4.pkl")
joblib.dump(final_features, STAGE2_OUT / "stage2_final_features_v4.pkl")

manifest = {
    "master_v4": str(OUT_MASTER_V4),
    "temporal_encoders_v4": str(temporal_out_file),
    "temporal_encoders_final_v4": str(STAGE2_OUT / "temporal_encoders_final_v4.pkl"),
    "unstable_branches_v4": str(STAGE2_OUT / "unstable_branches_list_v4.json"),
    "scaler_v4": str(STAGE2_OUT / "stage2_scaler_v4.pkl"),
    "final_features": final_features,
    "candidate_features": candidate_features,
    "train_years": train_years,
    "val_year": 2023,
    "test_year": 2024,
    "rows_master": int(len(df))
}
with open(STAGE2_OUT / "stage2_final_manifest_v4.json","w",encoding="utf8") as f:
    json.dump(manifest, f, indent=2)
print("Saved manifest v4 ->", STAGE2_OUT / "stage2_final_manifest_v4.json")

# Print a small diagnostics summary for you to paste back
print("\n=== DIAGNOSTICS ===")
print("Rows (master):", len(df))
print("Unique Colleges:", df['College_Code'].nunique() if 'College_Code' in df.columns else "n/a")
print("Unique Branches:", df['Branch'].nunique() if 'Branch' in df.columns else "n/a")
print("Unique Categories:", df['Category'].nunique() if 'Category' in df.columns else "n/a")
print("Global train median (Cutoff_log1p):", global_median_train)
print("Sample columns added:", new_cols[:6])
print("Sample final feature columns (first 12):", final_features[:12])

# show sample rows for inspection (head)
display(df.head(6))

print("\nCELL 2 complete — causal features computed and saved (v4).")


Loaded sorted master: D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\stage2_outputs\df_stage2_sorted.csv shape: (212487, 40)
Loaded year_max_ranks for years: [2020, 2021, 2022, 2023, 2024]
Global target median (for fallback): 11.097970469988296
Candidate features prepared (count): 30
Final model-ready features (count): 30
Sample final features: ['College_Code_prev1_median_scaled', 'College_Code_prev3_median_scaled', 'College_Code_count_prev_scaled', 'College_Code_trend_prev3_slope_scaled', 'College_Code_prev1_median_ranknorm_scaled', 'College_Code_prev3_median_ranknorm_scaled', 'College_Code_enc_prev_filled_v4_scaled', 'Branch_prev1_median_scaled', 'Branch_prev3_median_scaled', 'Branch_count_prev_scaled', 'Branch_trend_prev3_slope_scaled', 'Branch_prev1_median_ranknorm_scaled']
Saved Stage-2 master with v4 encoders -> D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\stage2_outputs\KCET_stage2_final_with_encoders_v4.csv
Saved Tier1 v4 CSVs

,College_Code,College_Name,Category,Branch,Cutoff_Rank,Cutoff_log1p,Year,Round,Exam_Type,College_Tier,...,Category_trend_prev3_slope_scaled,Category_prev1_median_ranknorm_scaled,Category_prev3_median_ranknorm_scaled,Category_enc_prev_filled_v4_scaled,college_pop_prev_filled_scaled,branch_pop_prev_filled_scaled,college_pop_prev_norm_scaled,branch_pop_prev_norm_scaled,Rank_norm_scaled,Rank_norm_log1p_scaled
0,E001,University Visveswariah College of Engineering...,1G,CE Civil,29057.0,10.277049,2020,1,KCET,1,...,-0.494624,-4.074675,-3.995456,0.302936,-0.897569,-0.844614,-0.897569,-0.844614,-0.837366,-0.839736
1,E001,University Visveswariah College of Engineering...,1K,CE Civil,30427.0,10.323119,2020,1,KCET,1,...,-0.494624,-4.074675,-3.995456,0.302936,-0.897569,-0.844614,-0.897569,-0.844614,-0.803235,-0.798517
2,E001,University Visveswariah College of Engineering...,2AG,CE Civil,27430.0,10.219429,2020,1,KCET,1,...,-0.494624,-4.074675,-3.995456,0.302936,-0.897569,-0.844614,-0.897569,-0.844614,-0.877900,-0.889092
3,E001,University Visveswariah College of Engineering...,2AR,CE Civil,28173.0,10.246155,2020,1,KCET,1,...,-0.494624,-4.074675,-3.995456,0.302936,-0.897569,-0.844614,-0.897569,-0.844614,-0.859390,-0.866498
4,E001,University Visveswariah College of Engineering...,2BG,CE Civil,26255.0,10.175650,2020,1,KCET,1,...,-0.494624,-4.074675,-3.995456,0.302936,-0.897569,-0.844614,-0.897569,-0.844614,-0.907174,-0.925014
5,E001,University Visveswariah College of Engineering...,2BR,CE Civil,50152.0,10.822834,2020,1,KCET,1,...,-0.494624,-4.074675,-3.995456,0.302936,-0.897569,-0.844614,-0.897569,-0.844614,-0.311816,-0.236867



CELL 2 complete — causal features computed and saved (v4).
